#### first code cell runs a terminal command (!pip install tensorflow) to ensure the TensorFlow library is installed in the environment. The second cell imports all the necessary Python libraries needed to build the project. This includes tools from Keras for data processing (Tokenizer, pad_sequences, to_categorical), tools to build the neural networks (Sequential, Embedding, LSTM, GRU, Bidirectional), Scikit-learn for splitting the data (train_test_split), and standard libraries like time, numpy, and pandas for mathematical operations and data formatting.

In [ ]:
!pip install tensorflow    

: 

In [2]:
import time
import numpy as np
import pandas as pd

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, GRU, Bidirectional, Dense, Dropout
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

#### This code cell creates a multi-line string variable named faqs that stores the entire text dataset consisting of frequently asked questions about Dr. Reddy's Foundation. The code then uses print(faqs[:500]) to display the first 500 characters of the text on the screen, just to verify that the string has been loaded into memory correctly

In [3]:
faqs = """About Dr. Reddy's Foundation
What is Dr. Reddy's Foundation?
Dr. Reddy's Foundation is a not for profit organization founded in 1996 that works to empower socially and economically disadvantaged communities across India.
When was Dr. Reddy's Foundation established?
The foundation was established in the year 1996 and is headquartered in Hyderabad Telangana.
What is the vision of Dr. Reddy's Foundation?
The vision is to enable sustainable social impact at scale.
What is the mission of Dr. Reddy's Foundation?
The mission is to empower communities through improved education health livelihood and climate action outcomes.
In how many states does the foundation operate?
The foundation currently operates in more than twenty states and union territories across India.
How many people has the foundation impacted so far?
The foundation has directly impacted well over twenty five lakh people since it was founded.
What are the main focus areas of the foundation?
The main focus areas are education healthcare livelihoods skill development inclusion and climate action.
Education Programs
What education initiatives does the foundation run?
The foundation runs neighborhood schools a vocational junior college and programs that improve learning outcomes in government schools.
What is Kallam Anji Reddy Vidyalaya?
Kallam Anji Reddy Vidyalaya is a school set up by the foundation in Chandanagar Hyderabad in 2001 for children from low income families.
What is the School Improvement Program?
The School Improvement Program supports government schools with infrastructure remedial education and teacher training.
How many students have benefited from the education programs?
Over three lakh fifty thousand students across more than two hundred schools have benefited with about half of them being girls.
Skill Development and Livelihoods
What is LABS?
LABS stands for Livelihood Advancement Business School and is the flagship short term skill training and job placement program of the foundation.
When did the skill development program start?
The skill development interventions of the foundation started in the year 1999.
How long is the LABS training program?
The LABS program traditionally offers ninety two days of technical training along with life skills training and job placement support.
How many youth have been trained under LABS?
More than two lakh ninety thousand youth have been trained under the LABS program so far.
Does the foundation support persons with disabilities?
Yes the foundation runs a dedicated LABS program for persons with disabilities across multiple states.
What is Skilling Rural India?
Skilling Rural India is an initiative that helps rural youth gain employability skills and entrepreneurship opportunities.
Rural Livelihoods Healthcare and Climate Action
What does the rural livelihood program do?
The rural livelihood program supports small and marginal farmers with advanced farming practices technology use and alternate income through livestock.
Does the foundation work on healthcare?
Yes the foundation works to strengthen public health systems as part of its broader social impact goals.
Does the foundation work on climate action?
Yes climate action is one of the core focus areas of the foundation alongside education health and livelihood.
Getting Involved
Where is Dr. Reddy's Foundation headquartered?
The foundation is headquartered in Hyderabad Telangana India.
Is the foundation linked to Dr. Reddy's Laboratories?
Yes the foundation is the not for profit CSR partner associated with the Dr Reddy's group.
How can someone support the foundation?
People can support the foundation by donating volunteering or partnering with it through corporate social responsibility programs.
Where can I find more information?
You can visit the official website of Dr. Reddy's Foundation to read the latest annual report and program details.
"""
print(faqs[:500])

About Dr. Reddy's Foundation
What is Dr. Reddy's Foundation?
Dr. Reddy's Foundation is a not for profit organization founded in 1996 that works to empower socially and economically disadvantaged communities across India.
When was Dr. Reddy's Foundation established?
The foundation was established in the year 1996 and is headquartered in Hyderabad Telangana.
What is the vision of Dr. Reddy's Foundation?
The vision is to enable sustainable social impact at scale.
What is the mission of Dr. Reddy's 


#### code initializes an empty Keras Tokenizer object and then uses the fit_on_texts([faqs]) function to scan the entire dataset. During this scan, the Tokenizer builds a dictionary where every unique word in the text is assigned a specific integer ID (with the most frequent words getting lower numbers). The code then calculates a variable called vocab_size by taking the total number of unique words and adding 1, which reserves the ID 0 specifically for padding purposes later on

In [4]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([faqs])

vocab_size = len(tokenizer.word_index) + 1
print("Unique words found:", len(tokenizer.word_index))
print("vocab_size (incl. reserved padding id 0):", vocab_size)

Unique words found: 225
vocab_size (incl. reserved padding id 0): 226


#### This cell creates an empty list named input_sequences. A for loop splits the FAQ text line-by-line, and texts_to_sequences converts the English words in each line into their newly assigned integer IDs. A nested inner loop for i in range(1, len(tokenized_sentence)) creates growing slices of that sequence (e.g., the first 2 words, then the first 3 words, etc.) and appends each slice to input_sequences. This generates hundreds of individual training examples where a sequence of words leads up to a final target word

In [5]:
input_sequences = []
for sentence in faqs.split('\n'):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[:i + 1])

print("Total training rows generated:", len(input_sequences))
print("First 5 rows (raw id lists, not yet padded):")
for row in input_sequences[:5]:
    print(row)

Total training rows generated: 526
First 5 rows (raw id lists, not yet padded):
[54, 7]
[54, 7, 8]
[54, 7, 8, 2]
[9, 3]
[9, 3, 7]


#### code uses a list comprehension to calculate max_len, which is the length of the single longest sequence generated in the previous step. It then passes the entire list of training rows into the pad_sequences function with the argument padding='pre'. This converts the list into a uniform 2D matrix by adding zeroes to the left side of any sequence that is shorter than max_len, ensuring every single row is exactly the same length

In [6]:
max_len = max([len(x) for x in input_sequences])
print("max_len (length of the longest sequence):", max_len)

padded_input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')
print("padded_input_sequences shape:", padded_input_sequences.shape)
print("Same first 5 rows, now padded:")
print(padded_input_sequences[:5])

max_len (length of the longest sequence): 23
padded_input_sequences shape: (526, 23)
Same first 5 rows, now padded:
[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 54  7]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 54  7  8]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 54  7  8  2]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  9  3]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  9  3  7]]


#### Using NumPy array slicing ([:, :-1]), the code strips away the very last column of the padded matrix and assigns the rest to the variable X; this represents the input context the model will read. It then slices exactly the last column ([:, -1]) and assigns it to y, representing the target word the model must predict. Finally, it uses the to_categorical function to convert the integer values in y into one-hot encoded vectors, transforming the target into a probability distribution format matching the total vocab_size

In [7]:
X = padded_input_sequences[:, :-1]
y = padded_input_sequences[:, -1]

y = to_categorical(y, num_classes=vocab_size)

print("X shape:", X.shape)   # (num_rows, max_len - 1)
print("y shape:", y.shape)   # (num_rows, vocab_size)

X shape: (526, 22)
y shape: (526, 226)


#### This cell utilizes the train_test_split function to randomly divide the X and y data matrices, assigning 85% of the rows for training the model and reserving 15% as unseen testing data (test_size=0.15). It also defines a variable input_len by subtracting 1 from max_len, which formally tells the upcoming neural networks exactly how many timesteps of input they should expect per sequence

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

input_len = max_len - 1  # length of every row in X

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("input_len (timesteps the model reads per example):", input_len)

X_train: (447, 22)  X_test: (79, 22)
input_len (timesteps the model reads per example): 22


#### code defines a custom function to build a Keras Sequential model containing an Input layer, an Embedding layer to turn IDs into dense vectors, two stacked LSTM layers (with 0.1 dropout to reduce overfitting), and a Dense layer with a softmax activation to output word probabilities. The model is compiled with the Adam optimizer and categorical crossentropy loss. A separate cell sets up an EarlyStopping callback to halt training if validation loss stops improving. The final cell uses model_lstm.fit to train the network on the X_train data for 100 epochs, measures the exact time it takes using the time module, and calculates the final accuracy on the unseen test data

In [9]:
def build_sequential_lstm():
    model = Sequential([
        Input(shape=(input_len,)),
        Embedding(vocab_size, 100),
        LSTM(150,dropout=0.1),
        # LSTM(150, dropout=0.1),
        Dense(vocab_size, activation='softmax',kernel_regularizer='l2'),
    ], name="Sequential_LSTM")
    return model


model_lstm = build_sequential_lstm()
model_lstm.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.summary()

Model: "Sequential_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 22, 100)        │        22,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 226)            │        34,126 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 207,326 (809.87 KB)

 Trainable params: 207,326 (809.87 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [11]:
t0 = time.time()
history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    verbose=1,
    callbacks=[early_stop]
)
train_time_lstm = time.time() - t0
test_loss_lstm, test_acc_lstm = model_lstm.evaluate(X_test, y_test, verbose=0)
print(f"\nSequential LSTM — training time: {train_time_lstm:.1f}s | test accuracy: {test_acc_lstm:.4f}")

Epoch 1/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.0515 - loss: 6.9488 - val_accuracy: 0.0380 - val_loss: 6.5356
Epoch 2/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0694 - loss: 6.2544 - val_accuracy: 0.0380 - val_loss: 6.2806
Epoch 3/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0872 - loss: 5.8655 - val_accuracy: 0.0380 - val_loss: 6.2611
Epoch 4/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0626 - loss: 5.6611 - val_accuracy: 0.0633 - val_loss: 6.1145
Epoch 5/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0940 - loss: 5.4847 - val_accuracy: 0.0380 - val_loss: 6.0332
Epoch 6/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0962 - loss: 5.3447 - val_accuracy: 0.0633 - val_loss: 5.9578
Epoch 7/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0917 - loss: 5.2073 - val_accuracy: 0.0380 - val_loss: 5.8946
Epoch 8/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1208 - loss: 5.0848 - val_accuracy: 0.

#### This section builds a second neural network, but replaces the standard LSTM layers with Bidirectional(LSTM(...)) layers. This architectural change forces the network to process the sequence of input words in both a forward (left-to-right) and backward (right-to-left) direction simultaneously before concatenating the results. Like the first model, the code compiles the network, prints a summary showing a massive increase in trainable parameters, trains it using fit for 100 epochs, and measures the training time and test accuracy

In [12]:
def build_bidirectional_lstm():
    model = Sequential([
        Input(shape=(input_len,)),
        Embedding(vocab_size, 100),
        Bidirectional(LSTM(150, return_sequences=True)),
        Bidirectional(LSTM(150)),
        Dense(vocab_size, activation='softmax'),
    ], name="Bidirectional_LSTM")
    return model


model_bilstm = build_bidirectional_lstm()
model_bilstm.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_bilstm.summary()

Model: "Bidirectional_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 22, 100)        │        22,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 22, 300)        │       301,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 300)            │       541,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 226)            │        68,026 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 933,026 (3.56 MB)

 Trainable params: 933,026 (3.56 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
t0 = time.time()
history_bilstm = model_bilstm.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    verbose=1,
)
train_time_bilstm = time.time() - t0
test_loss_bilstm, test_acc_bilstm = model_bilstm.evaluate(X_test, y_test, verbose=0)
print(f"\nBidirectional LSTM — training time: {train_time_bilstm:.1f}s | test accuracy: {test_acc_bilstm:.4f}")

Epoch 1/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 79ms/step - accuracy: 0.0425 - loss: 5.2852 - val_accuracy: 0.0380 - val_loss: 5.2610
Epoch 2/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.0626 - loss: 4.9296 - val_accuracy: 0.0506 - val_loss: 5.5152
Epoch 3/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.0694 - loss: 4.8143 - val_accuracy: 0.0380 - val_loss: 5.5626
Epoch 4/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.0872 - loss: 4.7131 - val_accuracy: 0.0380 - val_loss: 5.7693
Epoch 5/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.0828 - loss: 4.5893 - val_accuracy: 0.0506 - val_loss: 5.8777
Epoch 6/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.0716 - loss: 4.4875 - val_accuracy: 0.0506 - val_loss: 5.9821
Epoch 7/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.0895 - loss: 4.3490 - val_accuracy: 0.0506 - val_loss: 6.0184
Epoch 8/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.0984 - loss: 4.2394 - val_accuracy: 0.

#### code defines a third model function, this time swapping the LSTM components entirely for GRU (Gated Recurrent Unit) layers. Because GRUs have fewer internal gates, the printed model summary shows significantly fewer trainable parameters than the Bidirectional model. The code compiles the model, runs the fit function over the training data for 100 epochs, records the elapsed training time, and evaluates the model's accuracy on the held-out test set

In [14]:
def build_gru():
    model = Sequential([
        Input(shape=(input_len,)),
        Embedding(vocab_size, 100),
        GRU(150, return_sequences=True),
        GRU(150),
        Dense(vocab_size, activation='softmax'),
    ], name="GRU")
    return model


model_gru = build_gru()
model_gru.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_gru.summary()

Model: "GRU"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 22, 100)        │        22,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 22, 150)        │       113,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 150)            │       135,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 226)            │        34,126 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 306,026 (1.17 MB)

 Trainable params: 306,026 (1.17 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
t0 = time.time()
history_gru = model_gru.fit(
    X_train, y_train,
    epochs=50,
    validation_data=(X_test, y_test),
    verbose=1,
)
train_time_gru = time.time() - t0
test_loss_gru, test_acc_gru = model_gru.evaluate(X_test, y_test, verbose=0)
print(f"\nGRU — training time: {train_time_gru:.1f}s | test accuracy: {test_acc_gru:.4f}")

Epoch 1/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.0671 - loss: 5.3509 - val_accuracy: 0.0380 - val_loss: 5.2818
Epoch 2/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.0716 - loss: 4.9839 - val_accuracy: 0.0506 - val_loss: 5.2751
Epoch 3/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.0559 - loss: 4.8482 - val_accuracy: 0.0380 - val_loss: 5.5479
Epoch 4/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.1007 - loss: 4.7424 - val_accuracy: 0.0506 - val_loss: 5.6251
Epoch 5/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.1141 - loss: 4.6471 - val_accuracy: 0.0633 - val_loss: 5.7030
Epoch 6/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.1409 - loss: 4.5009 - val_accuracy: 0.0380 - val_loss: 5.6833
Epoch 7/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.1365 - loss: 4.3567 - val_accuracy: 0.0759 - val_loss: 5.8752
Epoch 8/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.1611 - loss: 4.1696 - val_accuracy: 0.1013 - v

#### code defines a helper function called avg_predict_time_ms that takes a single sample from the test data and asks a model to run a .predict() pass multiple times to calculate the average milliseconds required for inference. The code then constructs a structured Pandas DataFrame (a table) by passing in a list of dictionaries. Each dictionary extracts the total parameters, final training accuracy, final test accuracy, test loss, total training time, and inference speed for each of the three trained models, printing them side-by-side for easy visual comparison

In [16]:
def avg_predict_time_ms(model, X_sample, repeats=10):
    sample = X_sample[:1]
    model.predict(sample, verbose=0)  # warm-up call, not timed
    t0 = time.time()
    for _ in range(repeats):
        model.predict(sample, verbose=0)
    return (time.time() - t0) / repeats * 1000


comparison = pd.DataFrame([
    {
        "Model": "Sequential LSTM",
        "Params": model_lstm.count_params(),
        "Train Acc": history_lstm.history['accuracy'][-1],
        "Val/Test Acc": test_acc_lstm,
        "Test Loss": test_loss_lstm,
        "Train Time (s)": round(train_time_lstm, 1),
        "Avg Predict Time (ms)": round(avg_predict_time_ms(model_lstm, X_test), 2),
    },
    {
        "Model": "Bidirectional LSTM",
        "Params": model_bilstm.count_params(),
        "Train Acc": history_bilstm.history['accuracy'][-1],
        "Val/Test Acc": test_acc_bilstm,
        "Test Loss": test_loss_bilstm,
        "Train Time (s)": round(train_time_bilstm, 1),
        "Avg Predict Time (ms)": round(avg_predict_time_ms(model_bilstm, X_test), 2),
    },
    {
        "Model": "GRU",
        "Params": model_gru.count_params(),
        "Train Acc": history_gru.history['accuracy'][-1],
        "Val/Test Acc": test_acc_gru,
        "Test Loss": test_loss_gru,
        "Train Time (s)": round(train_time_gru, 1),
        "Avg Predict Time (ms)": round(avg_predict_time_ms(model_gru, X_test), 2),
    },
])

comparison

,Model,Params,Train Acc,Val/Test Acc,Test Loss,Train Time (s),Avg Predict Time (ms)
0,Sequential LSTM,207326,0.284116,0.126582,5.526820,9.6,51.59
1,Bidirectional LSTM,933026,0.935123,0.164557,9.084472,73.1,53.32
2,GRU,306026,0.930649,0.202532,7.111835,32.0,51.67


#### code cell defines a generate_text loop. For a requested number of words, it takes a raw English text string, uses the Tokenizer to convert it to IDs, pads it to the correct input_len, and feeds it into the model.predict function. It uses np.argmax to isolate the single highest-probability integer from the output, looks up the corresponding English word in the tokenizer.word_index, appends it to the original string, and restarts the loop. Finally, it runs this function for all three models using the exact same starting phrase ("Dr. Reddy's Foundation is a not ") and prints their unique outputs

In [17]:
def generate_text(model, seed_text, num_words=10):
    text = seed_text
    for _ in range(num_words):
        token_text = tokenizer.texts_to_sequences([text])[0]
        padded_token_text = pad_sequences([token_text], maxlen=input_len, padding='pre')
        pos = np.argmax(model.predict(padded_token_text, verbose=0))
        next_word = ""
        for word, index in tokenizer.word_index.items():
            if index == pos:
                next_word = word
                break
        if not next_word:
            break
        text = text + " " + next_word
    return text


seed = "Dr. Reddy's Foundation is a not "
for name, model in [("Sequential LSTM", model_lstm),
                     ("Bidirectional LSTM", model_bilstm),
                     ("GRU", model_gru)]:
    t0 = time.time()
    generated = generate_text(model, seed, num_words=8)
    elapsed = time.time() - t0
    print(f"[{name}] ({elapsed:.2f}s for 8 words)\n  {generated}\n")

[Sequential LSTM] (0.42s for 8 words)
  Dr. Reddy's Foundation is a not  in the year in and and and and

[Bidirectional LSTM] (0.42s for 8 words)
  Dr. Reddy's Foundation is a not  for profit organization founded in 1996 that works

[GRU] (0.42s for 8 words)
  Dr. Reddy's Foundation is a not  for profit organization founded in 1996 that works

